In [1]:
import pandas as pd
import re
from statistics import median

# ========================
# Config
# ========================
INPUT_CSV = "Manual Data Correction - test.csv"
OUTPUT_CSV = "Manual Data Correction - test_with_comparative_tags_reordered_7.csv"

VARIANT_COLS = [
    "targets",
    "corrected_targets",
    "corrected_targets_alt1",
    "corrected_targets_alt2",
    "corrected_targets_alt3",
]

TRIPLET_RE = re.compile(r"\[A\](.*?)\[O\](.*?)\[S\](.*)", flags=re.UNICODE)
PUNCT_RE = re.compile(r"[^\w\s]", flags=re.UNICODE)

# ========================
# Parsing & Normalization
# ========================
def norm_text(s: str) -> str:
    s = s.strip().lower()
    s = re.sub(r"\s+", " ", s)
    return s

def parse_block(text):
    """Return list of normalized (aspect, opinion, sentiment) triplets."""
    trips = []
    if pd.isna(text) or str(text).strip() == "":
        return trips
    for line in str(text).split("\n"):
        m = TRIPLET_RE.search(line.strip())
        if not m:
            continue
        a, o, s = norm_text(m.group(1)), norm_text(m.group(2)), norm_text(m.group(3))
        trips.append((a, o, s))
    return trips

def word_count(s: str) -> int:
    return len([w for w in re.findall(r"[^\s]+", s)])

# ========================
# Typo Detection Helpers
# ========================
def typo_signals(s: str) -> int:
    """Crude cleanliness heuristic: doubled letters/spaces, stray punctuation spacing."""
    score = 0
    if re.search(r"(ii|ss|tt|aa|uu)", s): score += 1
    if "  " in s: score += 1
    if re.search(r"\s+[.,;:]\s*", s): score += 1
    return score

def levenshtein(a: str, b: str) -> int:
    """Simple dynamic-programming Levenshtein distance."""
    if a == b: return 0
    if len(a) < len(b): a, b = b, a
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            ins = cur[-1] + 1
            dele = prev[j] + 1
            sub = prev[j-1] + (ca != cb)
            cur.append(min(ins, dele, sub))
        prev = cur
    return prev[-1]

def looks_like_typo_pair(a: str, b: str) -> int:
    """
    Return +1 if b looks cleaner (typo_corrected), -1 if a looks cleaner,
    0 if undecided/not a small edit.
    """
    dist = levenshtein(a, b)
    max_len = max(len(a), len(b))
    thresh = 2 if max_len <= 12 else 3  # allow slightly larger for longer strings
    if dist == 0 or dist > thresh:
        return 0

    sig_a, sig_b = typo_signals(a), typo_signals(b)
    alpha_ratio_a = len(re.findall(r"[0-9A-Za-zá-ž]", a)) / max(1, len(a))
    alpha_ratio_b = len(re.findall(r"[0-9A-Za-zá-ž]", b)) / max(1, len(b))

    # cleaner = fewer signals, higher alphanumeric ratio
    score_a = (-sig_a, alpha_ratio_a)
    score_b = (-sig_b, alpha_ratio_b)
    if score_b > score_a: return +1
    if score_a > score_b: return -1
    return 0

def _norm_for_match(s: str) -> str:
    """Normalize for substring match against input."""
    s = s.lower().strip()
    s = re.sub(r"[^\w\s]", " ", s)
    s = re.sub(r"\s+", " ", s)
    return s


def _contains_at_least_k(sub_ops, big_ops, k=2):
    """
    Do any opinion(s) in big_ops contain at least k distinct items from sub_ops?
    Opinions are already normalized.
    """
    sub_set = set(x for x in sub_ops if x)
    if not sub_set:
        return False
    for b in big_ops:
        hits = sum(1 for a in sub_set if a and a in b)
        if hits >= k:
            return True
    return False

def _typo_threshold(a: str, b: str) -> int:
    max_len = max(len(a), len(b))
    return 2 if max_len <= 12 else 3

def strip_punct_keep_space(s: str) -> str:
    return re.sub(r"[^\w\s]", " ", s)

def norm_no_punct(s: str) -> str:
    s = strip_punct_keep_space(s.lower().strip())
    s = re.sub(r"\s+", " ", s)
    return s

def tokens(s: str):
    return norm_no_punct(s).split()

def jaccard_overlap(a_tokens, b_tokens) -> float:
    A, B = set(a_tokens), set(b_tokens)
    if not A and not B:
        return 1.0
    if not A or not B:
        return 0.0
    return len(A & B) / len(A | B)

def tiny_typo_threshold(a: str, b: str) -> int:
    # distance threshold after punctuation removed
    L = max(len(a), len(b))
    return 2 if L <= 12 else 3


# ========================
# Aspect Tokens & Superset Rule
# ========================
def norm_tokens_aspect(s: str):
    s = s.lower().strip()
    s = PUNCT_RE.sub(" ", s)
    s = re.sub(r"\s+", " ", s)
    toks = s.split()
    # simple Indonesian '-nya' stripper
    cleaned = [(t[:-3] if t.endswith("nya") and len(t) > 3 else t) for t in toks]
    return tuple(cleaned)

def is_strict_superset(long_toks, short_toks):
    set_long, set_short = set(long_toks), set(short_toks)
    return len(set_long) > len(set_short) and set_short.issubset(set_long)

# ========================
# Features for *_tags_detail
# ========================
def features(trips):
    if not trips: return None
    n = len(trips)
    op_lens = [word_count(o) for (_, o, _) in trips]
    as_lens = [word_count(a) for (a, _, _) in trips]
    return dict(
        n_triplets=n,
        max_opinion_len=max(op_lens) if op_lens else 0,
        min_opinion_len=min(op_lens) if op_lens else 0,
        max_aspect_len=max(as_lens) if as_lens else 0,
        min_aspect_len=min(as_lens) if as_lens else 0,
    )

def detail_string(feats: dict) -> str:
    # keep your multi-line style
    return ";\n".join(f"{k}={v}" for k, v in feats.items())

# ========================
# Comparative Tagging (OPINION)
# ========================
def _norm_for_opinion_cmp(s: str) -> str:
    # normalize for substring containment comparisons
    s = s.lower().strip()
    s = re.sub(r"[^\w\s]", " ", s)  # remove punctuation
    s = re.sub(r"\s+", " ", s)
    return s

def tag_long_vs_split_opinion(variants):
    """
    Merge/split detector with typo guard:
      - If at an anchor any cross-variant pair has tiny char-distance (punctuation-free),
        treat it as potential typo and SKIP long/split at that anchor.
      - Else, detect merge/split via containment (≥2 sub-opinions contained in one longer opinion).
      - If no anchor hits, fallback to global (n_triplets, then max_opinion_len).
    """
    labels = {v: set() for v in variants}

    # Build (aspect, sentiment) -> per-variant list of opinions (raw, then we norm as needed)
    idx = {}
    for vname, trips in variants.items():
        per = {}
        for (a, o, s) in trips:
            per.setdefault((a, s), []).append(o)
        for key, ops in per.items():
            idx.setdefault(key, {})[vname] = ops

    found_anchor_pattern = False

    for key, per in idx.items():
        if len(per) < 2:
            continue
        items = list(per.items())
        # --- Typo guard: if any minimal pair at this anchor is under tiny threshold, skip this anchor ---
        typo_like = False
        for i in range(len(items)):
            vi, ops_i = items[i]
            for j in range(i+1, len(items)):
                vj, ops_j = items[j]
                best = _minimal_pair(ops_i, ops_j)
                if not best: 
                    continue
                dmin, ai, bj = best
                if dmin > 0 and dmin <= tiny_typo_threshold(norm_no_punct(ai), norm_no_punct(bj)):
                    typo_like = True
                    break
            if typo_like: break
        if typo_like:
            continue  # do not tag long/split on this anchor

        # --- Merge/split detection (permutation/subset) ---
        for i in range(len(items)):
            vi, ops_i = items[i]
            n_i = [_norm_for_opinion_cmp(x) for x in ops_i]
            for j in range(i+1, len(items)):
                vj, ops_j = items[j]
                n_j = [_norm_for_opinion_cmp(x) for x in ops_j]

                def contains_at_least_k(a_ops, b_ops, k=2):
                    for b in b_ops:
                        hits = sum(1 for a in set(a_ops) if a and a in b)
                        if hits >= k:
                            return True
                    return False

                a_splits_b_merges = (len(n_i) >= 2 and contains_at_least_k(n_i, n_j, k=2))
                b_splits_a_merges = (len(n_j) >= 2 and contains_at_least_k(n_j, n_i, k=2))

                if a_splits_b_merges and not b_splits_a_merges:
                    labels[vi].add("split_opinion"); labels[vj].add("long_opinion"); found_anchor_pattern = True
                elif b_splits_a_merges and not a_splits_b_merges:
                    labels[vj].add("split_opinion"); labels[vi].add("long_opinion"); found_anchor_pattern = True

    if found_anchor_pattern:
        return labels

    # ---------- Fallback: global rule ----------
    feats = {}
    for name, trips in variants.items():
        if not trips: continue
        n = len(trips)
        max_op = 0
        for (_, o, _) in trips:
            max_op = max(max_op, len([w for w in re.findall(r"[^\s]+", o)]))
        feats[name] = {"n": n, "max_op": max_op}

    if len(feats) <= 1:
        return labels

    ns = [f["n"] for f in feats.values()]
    n_min, n_max = min(ns), max(ns)
    if n_max > n_min:
        for name, f in feats.items():
            if f["n"] == n_max: labels[name].add("split_opinion")
            if f["n"] == n_min: labels[name].add("long_opinion")

    unlabeled = [name for name in variants if "split_opinion" not in labels[name] and "long_opinion" not in labels[name]]
    if len(unlabeled) >= 2:
        group = {name: feats[name] for name in unlabeled if name in feats}
        if len(group) >= 2:
            ops = [f["max_op"] for f in group.values()]
            op_min, op_max = min(ops), max(ops)
            if op_max > op_min:
                for name, f in group.items():
                    if f["max_op"] == op_max: labels[name].add("long_opinion")
                    if f["max_op"] == op_min: labels[name].add("split_opinion")

    return labels



# ========================
# Comparative Tagging (ASPECT)
# ========================
def tag_long_aspect_superset(variants):
    """
    long_aspect via strict superset rule with a guard:
      - Anchor = (opinion, sentiment).
      - First require that aspect sets actually differ for the anchor.
      - Then, if tokens(a) ⊂ tokens(b) (strict), tag the longer side as long_aspect.
    Returns dict[name -> bool]
    """
    long_marks = {v: False for v in variants}

    # (opinion, sentiment) -> per-variant set/list of aspect token tuples
    idx_aspects = {}
    for vname, trips in variants.items():
        per = {}
        for (a, o, s) in trips:
            key = (o, s)
            per.setdefault(key, []).append(norm_tokens_aspect(a))
        for key, toks in per.items():
            idx_aspects.setdefault(key, {})[vname] = toks

    for key, per in idx_aspects.items():
        if len(per) < 2:
            continue

        # Guard: only proceed if aspect sets differ across variants
        # (compare as stringified sets of token-tuples)
        sets_signature = set()
        for vname, toks in per.items():
            sets_signature.add(tuple(sorted(set(toks))))
        if len(sets_signature) <= 1:
            continue  # identical aspect sets -> no long_aspect

        vnames = list(per.keys())
        for i in range(len(vnames)):
            vi = vnames[i]
            for j in range(i+1, len(vnames)):
                vj = vnames[j]
                Ai = per[vi]
                Aj = per[vj]

                # If any ai is strict subset of any bj -> vj is long
                if any(is_strict_superset(bj, ai) for ai in Ai for bj in Aj):
                    long_marks[vj] = True
                # If any aj is strict subset of any bi -> vi is long
                if any(is_strict_superset(bi, aj) for aj in Aj for bi in Ai):
                    long_marks[vi] = True

    return long_marks


# ========================
# Comparative Tagging (TYPO) with INPUT
# ========================
def _minimal_pair(oplist1, oplist2):
    """Return (dist, x, y) for the minimal Levenshtein distance after punctuation stripped."""
    best = None
    for a in oplist1:
        na = norm_no_punct(a)
        for b in oplist2:
            nb = norm_no_punct(b)
            d = levenshtein(na, nb)
            if best is None or d < best[0]:
                best = (d, a, b)  # keep originals (pre-norm) for later checks
    return best

def _decide_typo_for_pair(a_raw, b_raw, norm_input):
    """
    Decide present/corrected for a single pair:
      - require small char distance on punctuation-free strings
      - require some token overlap (word-level sanity)
      - then: input presence -> cleanliness fallback
    Returns (+1 means b corrected & a present, -1 means a corrected & b present, 0 = skip)
    """
    a_np = norm_no_punct(a_raw)
    b_np = norm_no_punct(b_raw)
    d = levenshtein(a_np, b_np)
    if d == 0 or d > tiny_typo_threshold(a_np, b_np):
        return 0

    # word-level sanity check to avoid paraphrase / long-v-split
    tok_a, tok_b = tokens(a_raw), tokens(b_raw)
    if jaccard_overlap(tok_a, tok_b) < 0.4:
        return 0

    # input presence
    a_in = (a_np in norm_input)
    b_in = (b_np in norm_input)
    if a_in ^ b_in:
        return +1 if b_in else -1  # +1: b corrected / a present

    # cleanliness fallback
    sig_a, sig_b = typo_signals(a_np), typo_signals(b_np)
    alpha_a = len(re.findall(r"[0-9A-Za-zá-ž]", a_np)) / max(1, len(a_np))
    alpha_b = len(re.findall(r"[0-9A-Za-zá-ž]", b_np)) / max(1, len(b_np))
    score_a = (-sig_a, alpha_a)
    score_b = (-sig_b, alpha_b)
    if score_b > score_a:
        return +1
    if score_a > score_b:
        return -1
    return 0

def tag_typo_min_distance_using_input(variants, raw_input_text: str):
    """
    Typo tagging via punctuation-free minimal-distance pairing,
    for BOTH opinions and aspects, with word-overlap confirmation.
    """
    labels = {v: set() for v in variants}
    if not raw_input_text:
        return labels
    norm_input = norm_no_punct(raw_input_text)

    # Build two indices:
    # 1) (aspect, sentiment) -> opinions per variant
    opin_idx = {}
    # 2) (opinion, sentiment) -> aspects per variant
    as_idx = {}

    for vname, trips in variants.items():
        per_opin = {}
        per_as = {}
        for (a, o, s) in trips:
            per_opin.setdefault((a, s), []).append(o)
            per_as.setdefault((o, s), []).append(a)
        for k, v in per_opin.items():
            opin_idx.setdefault(k, {})[vname] = v
        for k, v in per_as.items():
            as_idx.setdefault(k, {})[vname] = v

    def _apply_min_pair(idx_map):
        for key, per in idx_map.items():
            if len(per) < 2:
                continue
            items = list(per.items())
            for i in range(len(items)):
                vi, list_i = items[i]
                for j in range(i+1, len(items)):
                    vj, list_j = items[j]
                    # Minimal pair on punctuation-free strings
                    best = _minimal_pair(list_i, list_j)
                    if not best:
                        continue
                    dmin, ai, bj = best
                    if dmin == 0 or dmin > tiny_typo_threshold(norm_no_punct(ai), norm_no_punct(bj)):
                        continue
                    # Avoid merge/split confusion: require decent token overlap
                    if jaccard_overlap(tokens(ai), tokens(bj)) < 0.4:
                        continue
                    # Decide tag directions
                    decision = _decide_typo_for_pair(ai, bj, norm_input)
                    if decision == +1:
                        labels[vj].add("typo_corrected"); labels[vi].add("typo_present")
                    elif decision == -1:
                        labels[vi].add("typo_corrected"); labels[vj].add("typo_present")

    # apply to opinions and to aspects
    _apply_min_pair(opin_idx)
    _apply_min_pair(as_idx)
    return labels



# ========================
# Variant Inclusion Rules (per your request)
# ========================
def select_variants_for_comparison_and_detail(row):
    """
    - If corrected_targets exists: compare only among corrected + alts; EXCLUDE 'targets' from comparison and details.
    - If corrected_targets missing and any alt exists: compare alts + targets (and include them in details).
    - If only targets exists (no corrected, no alts): NO comparison and NO details.
    Returns:
      variants_for_cmp: dict[name -> trips]
      variants_for_detail: dict[name -> trips]
    """
    present = {col: parse_block(row.get(col, "")) for col in VARIANT_COLS}
    has_corrected = bool(present["corrected_targets"])
    has_any_alt = any(bool(present[c]) for c in ["corrected_targets_alt1","corrected_targets_alt2","corrected_targets_alt3"])
    has_targets = bool(present["targets"])

    if has_corrected:
        # compare corrected + alts; exclude targets entirely
        include = {k: v for k, v in present.items() if k != "targets" and v}
        details = include.copy()  # exclude targets from details too
    elif (not has_corrected) and has_any_alt:
        # compare alts + targets
        include = {k: v for k, v in present.items() if (k == "targets" or k.startswith("corrected_targets_alt")) and v}
        details = include.copy()
    else:
        # only targets or nothing -> no comparison and no details
        include = {}
        details = {}

    return include, details


def _finalize_labels_for_variant(vname, variants_for_cmp, raw_triplets):
    """
    Enforce mutual exclusivity:
      - if both typo tags present -> keep only 'typo_corrected'
      - if both long/split present -> decide by n_triplets (max -> split, min -> long),
        tie-break by max_opinion_len (larger -> long, smaller -> split)
    """
    labs = set(raw_triplets)  # actually pass a set in
    # typo exclusivity
    if "typo_corrected" in labs and "typo_present" in labs:
        labs.discard("typo_present")  # keep corrected

    if "long_opinion" in labs and "split_opinion" in labs:
        # compute n_triplets & max_op_len for THIS variant
        trips = variants_for_cmp[vname]
        n = len(trips)
        max_op = max((len(tokens(o)) for (_, o, _) in trips), default=0)

        # compare to others
        ns = [len(variants_for_cmp[k]) for k in variants_for_cmp]
        n_min, n_max = min(ns), max(ns)
        # decide
        keep = None
        if n == n_max and n != n_min:
            keep = "split_opinion"
        elif n == n_min and n != n_max:
            keep = "long_opinion"
        else:
            # tie-break by max_opinion_len across peers
            peer_maxes = [max((len(tokens(o)) for (_, o, _) in variants_for_cmp[k]), default=0)
                          for k in variants_for_cmp]
            if max_op == max(peer_maxes):
                keep = "long_opinion"
            elif max_op == min(peer_maxes):
                keep = "split_opinion"
        # enforce
        if keep:
            labs = {x for x in labs if x not in ("long_opinion", "split_opinion")}
            labs.add(keep)
        else:
            # if truly ambiguous, drop both
            labs.discard("long_opinion")
            labs.discard("split_opinion")
    return labs


# ========================
# Run
# ========================
def main():
    df = pd.read_csv(INPUT_CSV)
    out_rows = []

    for _, row in df.iterrows():
        new_row = row.to_dict()

        # Decide which variants to compare and which to emit details for
        variants_for_cmp, variants_for_detail = select_variants_for_comparison_and_detail(row)

        # Initialize all tag columns
        for col in VARIANT_COLS:
            new_row[col + "_tags"] = ""
            new_row[col + "_tags_detail"] = ""

        # Fill details only for allowed variants (per rules)
        for col, trips in variants_for_detail.items():
            feats = features(trips)
            if feats:
                new_row[col + "_tags_detail"] = detail_string(feats)

        # If fewer than 2 variants in comparison → no comparative tags
        if len(variants_for_cmp) < 2:
            out_rows.append(new_row)
            continue

        # Comparative tags
        opin_labels = tag_long_vs_split_opinion(variants_for_cmp)
        aspect_long_marks = tag_long_aspect_superset(variants_for_cmp)
        typo_labels = tag_typo_min_distance_using_input(variants_for_cmp, raw_input_text=row.get("input", ""))

        # Emit labels only to variants participating in comparison
        for v in variants_for_cmp:
            labels = set()
            labels |= opin_labels.get(v, set())
            if aspect_long_marks.get(v, False):
                labels.add("long_aspect") 
            labels |= typo_labels.get(v, set())

            labels = _finalize_labels_for_variant(v, variants_for_cmp, labels)
            if labels:
                # keep your multi-line join
                new_row[v + "_tags"] = ",\n".join(sorted(labels))

        out_rows.append(new_row)

    df_out = pd.DataFrame(out_rows)

    # Reorder columns: core + per-variant (targets, tags, tags_detail) side-by-side
    tag_cols = [v + "_tags" for v in VARIANT_COLS]
    detail_cols = [v + "_tags_detail" for v in VARIANT_COLS]
    core_cols = [c for c in df_out.columns if c not in VARIANT_COLS + tag_cols + detail_cols]

    new_order = core_cols[:]
    for v in VARIANT_COLS:
        new_order.extend([v, v + "_tags", v + "_tags_detail"])

    df_out = df_out[new_order]
    df_out.to_csv(OUTPUT_CSV, index=False)
    print("✅ Saved:", OUTPUT_CSV)

if __name__ == "__main__":
    main()


✅ Saved: Manual Data Correction - test_with_comparative_tags_reordered_7.csv
